# Run this first in the notebook UI (path-independent installer)
import subprocess
import sys

packages = [
    "opentelemetry-api",
    "opentelemetry-sdk",
    "opentelemetry-exporter-otlp-proto-http",
    "python-dotenv",
    "PyYAML",
    "databricks-sql-connector",
]

base_cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
try:
    subprocess.check_call(base_cmd)
except subprocess.CalledProcessError:
    # macOS/Homebrew Python can be externally-managed (PEP 668)
    fallback_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--user",
        "--break-system-packages",
        *packages,
    ]
    subprocess.check_call(fallback_cmd)

print(f"Using Python: {sys.executable}")
print("Dependencies installed for active notebook kernel")

In [9]:
from pathlib import Path
import importlib
import os
import site
import subprocess
import sys

# Guard for kernels that don't pick up freshly installed packages immediately
if site.ENABLE_USER_SITE and site.getusersitepackages() not in sys.path:
    sys.path.append(site.getusersitepackages())

try:
    import yaml
    from dotenv import load_dotenv
except ModuleNotFoundError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--user",
            "--break-system-packages",
            "PyYAML",
            "python-dotenv",
        ]
    )
    importlib.invalidate_caches()
    if site.ENABLE_USER_SITE and site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())
    import yaml
    from dotenv import load_dotenv

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(project_root / ".env")

with open(project_root / "config" / "telemetry.emit.yaml", "r", encoding="utf-8") as f:
    TELEMETRY_CONFIG = yaml.safe_load(f)

required_env = [
    "DATABRICKS_HOST",
    "DATABRICKS_TOKEN",
    "DATABRICKS_CATALOG",
    "DATABRICKS_SCHEMA",
    "OTEL_SERVICE_NAME",
    "OTEL_SPANS_TABLE",
    "OTEL_LOGS_TABLE",
    "OTEL_METRICS_TABLE",
]

missing = [k for k in required_env if not os.getenv(k)]
if missing:
    raise ValueError(f"Missing required .env keys: {', '.join(missing)}")

DATABRICKS_HOST = os.environ["DATABRICKS_HOST"].rstrip("/")
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]
DATABRICKS_CATALOG = os.environ["DATABRICKS_CATALOG"]
DATABRICKS_SCHEMA = os.environ["DATABRICKS_SCHEMA"]
OTEL_SERVICE_NAME = os.environ["OTEL_SERVICE_NAME"]
OTEL_SPANS_TABLE = os.environ["OTEL_SPANS_TABLE"]
OTEL_LOGS_TABLE = os.environ["OTEL_LOGS_TABLE"]
OTEL_METRICS_TABLE = os.environ["OTEL_METRICS_TABLE"]

print("Loaded config for:")
print(f"  Workspace: {DATABRICKS_HOST}")
print(f"  Schema:    {DATABRICKS_CATALOG}.{DATABRICKS_SCHEMA}")
print(f"  Service:   {OTEL_SERVICE_NAME}")
print("  Tables:")
print(f"    spans:   {OTEL_SPANS_TABLE}")
print(f"    logs:    {OTEL_LOGS_TABLE}")
print(f"    metrics: {OTEL_METRICS_TABLE}")

Loaded config for:
  Workspace: https://e2-demo-field-eng.cloud.databricks.com
  Schema:    bx3.otel_demo
  Service:   otel-local-mac-emitter
  Tables:
    spans:   bx3.otel_demo.otel_spans_v2
    logs:    bx3.otel_demo.otel_logs_v2
    metrics: bx3.otel_demo.otel_metrics


In [10]:
import importlib
import logging
import site
import subprocess
import sys
import time
import urllib.error
import urllib.request

# Guard for kernels that do not automatically include user site-packages
if site.ENABLE_USER_SITE and site.getusersitepackages() not in sys.path:
    sys.path.append(site.getusersitepackages())

try:
    from opentelemetry import _logs as otel_logs
    from opentelemetry import metrics, trace
    from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
    from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk._logs import LoggerProvider, LoggingHandler
    from opentelemetry.sdk._logs.export import SimpleLogRecordProcessor
    from opentelemetry.sdk.metrics import MeterProvider
    from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor
except ModuleNotFoundError:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--user",
            "--break-system-packages",
            "opentelemetry-api",
            "opentelemetry-sdk",
            "opentelemetry-exporter-otlp-proto-http",
        ]
    )
    importlib.invalidate_caches()
    if site.ENABLE_USER_SITE and site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())

    from opentelemetry import _logs as otel_logs
    from opentelemetry import metrics, trace
    from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
    from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk._logs import LoggerProvider, LoggingHandler
    from opentelemetry.sdk._logs.export import SimpleLogRecordProcessor
    from opentelemetry.sdk.metrics import MeterProvider
    from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor

# Fixed endpoints only (no fallback/probing logic)
TRACES_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/tracing/otel/v1/traces"
LOGS_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/tracing/otel/v1/logs"
METRICS_ENDPOINT = f"{DATABRICKS_HOST}/api/2.0/otel/v1/metrics"

resource_cfg = TELEMETRY_CONFIG.get("resource", {})
resource = Resource.create(
    {
        "service.name": OTEL_SERVICE_NAME,
        "service.version": resource_cfg.get("service_version", "0.1.0"),
        "deployment.environment": resource_cfg.get("deployment_environment", "local"),
    }
)

def make_headers(table_name: str):
    if not table_name:
        raise ValueError("Target table is required for X-Databricks-UC-Table-Name")
    return {
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "X-Databricks-UC-Table-Name": table_name,
        "X-Databricks-Workspace-Url": DATABRICKS_HOST,
    }

def smoke_check(name: str, url: str, table_name: str) -> None:
    req = urllib.request.Request(
        url=url,
        method="POST",
        data=b"",
        headers={
            **make_headers(table_name),
            "Content-Type": "application/x-protobuf",
        },
    )
    try:
        urllib.request.urlopen(req, timeout=8)
    except urllib.error.HTTPError as e:
        raise RuntimeError(
            f"{name} endpoint failed local smoke test: {url} -> {e.code} {e.reason}. "
            "If this works inside Databricks notebooks but fails locally, this workspace likely does not expose OTLP ingest externally."
        ) from e

# Explicit connectivity check before wiring exporters
smoke_check("traces", TRACES_ENDPOINT, OTEL_SPANS_TABLE)
smoke_check("logs", LOGS_ENDPOINT, OTEL_LOGS_TABLE)
smoke_check("metrics", METRICS_ENDPOINT, OTEL_METRICS_TABLE)

# Traces
span_exporter = OTLPSpanExporter(
    endpoint=TRACES_ENDPOINT,
    headers=make_headers(OTEL_SPANS_TABLE),
)
tracer_provider = TracerProvider(resource=resource)
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer(OTEL_SERVICE_NAME)

# Logs
log_exporter = OTLPLogExporter(
    endpoint=LOGS_ENDPOINT,
    headers=make_headers(OTEL_LOGS_TABLE),
)
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(SimpleLogRecordProcessor(log_exporter))
otel_logs.set_logger_provider(logger_provider)
otel_handler = LoggingHandler(level=logging.INFO, logger_provider=logger_provider)
otel_logger = logging.getLogger(OTEL_SERVICE_NAME)
otel_logger.handlers = []
otel_logger.addHandler(otel_handler)
otel_logger.setLevel(logging.INFO)

# Metrics
metrics_cfg = TELEMETRY_CONFIG.get("metrics", {})
metric_export_interval_ms = int(metrics_cfg.get("export_interval_ms", 5000))

metric_exporter = OTLPMetricExporter(
    endpoint=METRICS_ENDPOINT,
    headers=make_headers(OTEL_METRICS_TABLE),
)
metric_reader = PeriodicExportingMetricReader(
    metric_exporter,
    export_interval_millis=metric_export_interval_ms,
)
meter_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_meter_provider(meter_provider)
meter = metrics.get_meter(OTEL_SERVICE_NAME)

print("Exporters configured")
print(f"  traces endpoint:      {TRACES_ENDPOINT}")
print(f"  logs endpoint:        {LOGS_ENDPOINT}")
print(f"  metrics endpoint:     {METRICS_ENDPOINT}")
print(f"  metric interval ms:   {metric_export_interval_ms}")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Exporters configured
  traces/logs endpoint: https://e2-demo-field-eng.cloud.databricks.com/api/2.0/tracing/otel
  metrics endpoint:     https://e2-demo-field-eng.cloud.databricks.com/api/2.0/otel
  metric interval ms:   5000


In [11]:
# Emit traces from YAML
traces_cfg = TELEMETRY_CONFIG.get("traces", {})
if traces_cfg.get("enabled", True):
    print("Emitting traces...")
    sleep_s = float(traces_cfg.get("sleep_ms_between_spans", 0)) / 1000.0
    for span_cfg in traces_cfg.get("spans", []):
        name = span_cfg.get("name", "unnamed-span")
        attrs = span_cfg.get("attributes", {})
        with tracer.start_as_current_span(name, attributes=attrs) as span:
            for event in span_cfg.get("events", []):
                span.add_event(event.get("name", "event"), attributes=event.get("attributes", {}))
            if sleep_s > 0:
                time.sleep(sleep_s)
    print("  Trace emission complete")
else:
    print("Trace emission disabled")

# Emit logs from YAML
logs_cfg = TELEMETRY_CONFIG.get("logs", {})
if logs_cfg.get("enabled", True):
    print("Emitting logs...")
    for entry in logs_cfg.get("entries", []):
        level_name = str(entry.get("level", "INFO")).upper()
        level = getattr(logging, level_name, logging.INFO)
        body = entry.get("body", "otel log")
        attrs = entry.get("attributes", {})
        otel_logger.log(level, body, extra=attrs)
    print("  Log emission complete")
else:
    print("Log emission disabled")

# Emit metrics from YAML
metrics_cfg = TELEMETRY_CONFIG.get("metrics", {})
if metrics_cfg.get("enabled", True):
    print("Emitting metrics...")

    counters = {}
    for c in metrics_cfg.get("counters", []):
        counters[c["name"]] = meter.create_counter(
            c["name"],
            description=c.get("description", ""),
            unit=c.get("unit", "1"),
        )
        for add in c.get("adds", []):
            counters[c["name"]].add(add.get("value", 1), add.get("attributes", {}))

    histograms = {}
    for h in metrics_cfg.get("histograms", []):
        histograms[h["name"]] = meter.create_histogram(
            h["name"],
            description=h.get("description", ""),
            unit=h.get("unit", "1"),
        )
        for record in h.get("records", []):
            histograms[h["name"]].record(record.get("value", 0.0), record.get("attributes", {}))

    verify_after_seconds = int(metrics_cfg.get("verify_after_seconds", 6))
    print(f"  Waiting {verify_after_seconds}s for periodic metric export...")
    time.sleep(verify_after_seconds)
    print("  Metric emission complete")
else:
    print("Metric emission disabled")

Emitting traces...


Failed to export span batch code: 404, reason: Not Found
Failed to export span batch code: 404, reason: Not Found


  Trace emission complete
Emitting logs...


Failed to export logs batch code: 404, reason: Not Found


  Log emission complete
Emitting metrics...
  Waiting 6s for periodic metric export...


Failed to export metrics batch code: 404, reason: Not Found


  Metric emission complete


Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found
Failed to export metrics batch code: 404, reason: Not Found


In [ ]:
# Flush and shutdown providers
for name, provider in [
    ("Traces", tracer_provider),
    ("Logs", logger_provider),
    ("Metrics", meter_provider),
]:
    provider.force_flush(timeout_millis=10000)
    print(f"{name} flushed")

tracer_provider.shutdown()
logger_provider.shutdown()
meter_provider.shutdown()
print("All providers shut down")

In [ ]:
# Optional verification via Databricks SQL Warehouse
warehouse_id = os.getenv("DATABRICKS_WAREHOUSE_ID", "").strip()
if not warehouse_id:
    print("DATABRICKS_WAREHOUSE_ID is empty; skipping SQL verification.")
else:
    from databricks import sql

    print("Running verification counts...")
    with sql.connect(
        server_hostname=DATABRICKS_HOST.replace("https://", ""),
        http_path=f"/sql/1.0/warehouses/{warehouse_id}",
        access_token=DATABRICKS_TOKEN,
    ) as conn:
        with conn.cursor() as cursor:
            for label, table_name in [
                ("spans", OTEL_SPANS_TABLE),
                ("logs", OTEL_LOGS_TABLE),
                ("metrics", OTEL_METRICS_TABLE),
            ]:
                cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
                count = cursor.fetchone()[0]
                print(f"  {label}: {table_name} -> {count}")